# Pro session — Qwen3-8B headroom + Control A timing

Two questions, one session:

1. Does Qwen3-8B clear the 70% headroom threshold?
2. What does the full Control A sweep actually cost in compute units?

Everything needed is written by this notebook. Nothing to upload.

**Threshold is unchanged: 70% on `strict`.** Amendment 001's rules are unchanged. Third model, same criteria.

### Before cell 1

`Runtime` → `Change runtime type` → **L4 GPU** (or A100 if offered). Not T4 — an 8B model in 16-bit needs ~16 GB and a T4 has 14.5.

### Note on the embedded code

Cells 3 and 4 write copies of `directions.py` and `harness.py`. Your repo stays the source of truth — if you change them there, re-copy. This is a one-off measurement session.

## Cell 1 — Which GPU did we get?

In [ ]:
import torch

if not torch.cuda.is_available():
    print(">>> STOP. Runtime > Change runtime type > L4 GPU > Save, then re-run.")
else:
    name = torch.cuda.get_device_name(0)
    free, total = torch.cuda.mem_get_info()
    print("Device :", name)
    print(f"Memory : {total/1e9:.1f} GB total, {free/1e9:.1f} GB free")
    if total / 1e9 < 20:
        print("\n>>> WARNING: under 20 GB. An 8B model in bf16 (~16 GB) will be tight.")
        print(">>> Try Runtime > Change runtime type > L4 or A100.")
    DTYPE = "bfloat16" if ("L4" in name or "A100" in name or "H100" in name) else "float16"
    print(f"\nUse --dtype {DTYPE}")
    print("Record the GPU name — you need its compute-unit rate for the cost estimate.")

## Cell 2 — Install and clone

In [ ]:
!pip -q install -U transformers accelerate
!git clone -q --depth 1 https://github.com/anthropics/jacobian-lens.git
!pip -q install -e jacobian-lens
!mkdir -p ablation && touch ablation/__init__.py

import jlens
print("jlens OK")

## Cell 3 — Write `directions.py`

In [ ]:
%%writefile ablation/directions.py
"""Direction selection and subspace projection for J-space / R-space ablation.

Implements the direction-set half of the ablation harness. Every selector here
produces a set of residual-stream directions of a specified size; the harness
projects them out. Keeping selection separate from projection is what makes the
matched controls of proposal 4.8 cheap: same projection, different selector.

Terminology (guide 2.1): nothing here is "the workspace". These are candidate
directions until Phase 3 says otherwise.
"""

from __future__ import annotations

from dataclasses import dataclass

import torch


# --- J-lens vector construction -------------------------------------------

def lens_vectors(
    unembed_weight: torch.Tensor, jacobian: torch.Tensor, token_ids: torch.Tensor
) -> torch.Tensor:
    """The J-lens vectors for ``token_ids`` at one layer.

    Paper §2.1 defines the J-lens vectors as the rows of ``W_U J_l``. Only the
    requested rows are materialised: the full product is ``[vocab, d_model]``
    and is far too large to hold for a real vocabulary.

    Args:
        unembed_weight: ``W_U``, shape ``[vocab, d_model]``.
        jacobian: ``J_l``, shape ``[d_model, d_model]``.
        token_ids: Shape ``[..., k]``.

    Returns:
        Shape ``[..., k, d_model]``.
    """
    rows = unembed_weight.index_select(0, token_ids.reshape(-1).to(unembed_weight.device))
    rows = rows.to(jacobian.dtype) @ jacobian
    return rows.reshape(*token_ids.shape, jacobian.shape[-1])


# --- Selectors -------------------------------------------------------------

def select_by_rank(
    lens_logits: torch.Tensor,
    k: int,
    *,
    rank_offset: int = 0,
    excluded: torch.Tensor | None = None,
) -> torch.Tensor:
    """Token ids ranked ``rank_offset .. rank_offset + k`` by lens score.

    ``rank_offset=0`` gives the top-k the paper ablates. ``rank_offset=k`` gives
    the next-k, which is the "matched but not selected" control: same lens, same
    size, adjacent rank band. A candidate subspace that matters no more than the
    next-k has not earned H1 (proposal 4.8, extended per the probe-swap design).

    Args:
        lens_logits: Shape ``[n_positions, vocab]``.
        k: Number of directions.
        rank_offset: Rank to start from.
        excluded: Boolean mask ``[n_positions, vocab]``; True entries are never
            selected. This carries the clean-pass exclusion — see
            :func:`clean_top_mask`.

    Returns:
        Shape ``[n_positions, k]``.
    """
    scores = lens_logits.clone()
    if excluded is not None:
        scores = scores.masked_fill(excluded, float("-inf"))
    top = scores.topk(rank_offset + k, dim=-1).indices
    return top[:, rank_offset:]


def random_lens_tokens(
    n_positions: int,
    k: int,
    vocab_size: int,
    generator: torch.Generator,
    *,
    excluded: torch.Tensor | None = None,
    device: torch.device | None = None,
) -> torch.Tensor:
    """Uniformly random token ids — matched-size random control (proposal 4.8).

    Drawn from the lens dictionary rather than isotropically, so the control
    asks "is it *these* lens directions, or any lens directions?". Strictly the
    harder question of the two; run both.
    """
    out = torch.empty(n_positions, k, dtype=torch.long, device=device)
    for p in range(n_positions):
        while True:
            cand = torch.randint(
                vocab_size, (k,), generator=generator, device=generator.device
            ).to(device)
            if excluded is None or not bool(excluded[p, cand].any()):
                out[p] = cand
                break
    return out


def random_isotropic(
    n_positions: int, k: int, d_model: int, generator: torch.Generator,
    *, device: torch.device | None = None, dtype: torch.dtype = torch.float32,
) -> torch.Tensor:
    """Isotropic random unit directions — the paper's random-direction control."""
    v = torch.randn(
        n_positions, k, d_model, generator=generator, device=generator.device,
        dtype=torch.float32,
    ).to(device=device, dtype=dtype)
    return v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)


def clean_top_mask(
    clean_next_token_logits: torch.Tensor, n_exclude: int = 10
) -> torch.Tensor:
    """Mask marking the clean pass's top-``n_exclude`` predictions per position.

    **This is the confound guard, and it is not optional.** Paper: "we do not
    ablate any tokens that appear in the top-10 tokens of a clean forward pass,
    so as to specifically target the J-space's effects on internal reasoning
    rather than report." Without it, ablation suppresses whatever the model was
    about to say, performance drops for a trivial reason, and H1 gets
    "confirmed" by an artifact.

    Args:
        clean_next_token_logits: Shape ``[n_positions, vocab]`` from an
            unablated forward pass.

    Returns:
        Boolean ``[n_positions, vocab]``, True where a token must not be ablated.
    """
    mask = torch.zeros_like(clean_next_token_logits, dtype=torch.bool)
    top = clean_next_token_logits.topk(n_exclude, dim=-1).indices
    return mask.scatter(-1, top, True)


# --- Projection ------------------------------------------------------------

@dataclass(frozen=True)
class Basis:
    """An orthonormal-row basis with a validity mask for rank-deficient sets."""

    rows: torch.Tensor   # [n_positions, k, d_model], orthonormal rows
    keep: torch.Tensor   # [n_positions, k], float 1/0
    rank: torch.Tensor   # [n_positions], effective rank actually removed


def orthonormalise(vectors: torch.Tensor, *, rtol: float = 1e-6) -> Basis:
    """Orthonormal basis for the span of each position's direction set.

    J-lens vectors are overcomplete and non-orthogonal (paper §2.3), so a set of
    k of them may span fewer than k dimensions. Small singular values are masked
    out rather than dropped, which keeps the operation batched and makes the
    effective rank observable — report it, because "we ablated k directions" is
    false if the span was smaller.
    """
    vectors = vectors.to(torch.float32)
    _, s, vh = torch.linalg.svd(vectors, full_matrices=False)
    keep = (s > rtol * s[..., :1].clamp_min(1e-30)).to(vectors.dtype)
    return Basis(rows=vh, keep=keep, rank=keep.sum(-1))


def project_out(
    hidden: torch.Tensor, basis: Basis, *, mode: str = "subspace"
) -> torch.Tensor:
    """Remove the component of ``hidden`` inside the spanned subspace.

    Args:
        hidden: Shape ``[n_positions, d_model]``.
        mode: ``"subspace"`` projects onto the orthogonal complement of the span
            in one step. ``"sequential"`` removes each direction in turn, which
            is order-dependent for non-orthogonal vectors and therefore removes
            *less* than the full span.

    The paper's phrasing — "zero out the residual stream's projection onto
    each" — does not disambiguate these, and for non-orthogonal J-lens vectors
    they differ. ``"subspace"`` is the default because it is the one that
    actually removes the content; ``"sequential"`` is provided so the choice can
    be tested rather than assumed. Record which was used.
    """
    h = hidden.to(torch.float32)
    if mode == "subspace":
        coeffs = torch.einsum("prd,pd->pr", basis.rows, h) * basis.keep
        return (h - torch.einsum("pr,prd->pd", coeffs, basis.rows)).to(hidden.dtype)
    if mode == "sequential":
        for i in range(basis.rows.shape[1]):
            v = basis.rows[:, i, :] * basis.keep[:, i : i + 1]
            h = h - (h * v).sum(-1, keepdim=True) * v
        return h.to(hidden.dtype)
    raise ValueError(f"unknown mode {mode!r}")

## Cell 4 — Write `harness.py`

In [ ]:
%%writefile ablation/harness.py
"""Two-pass ablation harness.

Pass 1 is a clean forward pass: it records the residual stream at every band
layer, computes lens logits, and captures the clean next-token distribution.
Pass 2 re-runs with the selected directions projected out.

Two passes are not an optimisation choice — the confound guard of proposal 4.4
(paper: exclude the clean pass's top-10) *requires* knowing the clean output
before choosing what to ablate.

This harness is built to Phase 3 requirements from the first line, per guide
§3a: Control A, Control B, and the Phase 3 sweep all run through it unchanged.
"""

from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import Any, Literal, Sequence

import torch

from jlens.hooks import ActivationRecorder
from jlens.lens import JacobianLens

from .directions import (
    Basis,
    clean_top_mask,
    lens_vectors,
    orthonormalise,
    project_out,
    random_isotropic,
    random_lens_tokens,
    select_by_rank,
)

Selector = Literal["topk", "next_k", "random_lens", "random_iso", "none"]


def record_at_or(spec, final):
    return sorted({*spec.layers, final})


def _mask_from_ids(ids: torch.Tensor, shape) -> torch.Tensor:
    """Rebuild the clean-top-k boolean mask from cached ids."""
    m = torch.zeros(shape, dtype=torch.bool, device=ids.device)
    return m.scatter(-1, ids, True)


@dataclass(frozen=True)
class AblationSpec:
    """One fully-specified ablation condition. Serialise this into every result.

    Attributes:
        layers: Band of block indices to ablate at. Light/medium/heavy differ
            only here — the paper varies the layer range, not k.
        k: Directions removed per position (proposal 4.7 sweeps this; the paper
            fixed it at 10). Sweeping k *and* layers multiplies runs — state
            which axis in prereg_phase3.md.
        selector: Which directions. ``"none"`` is the clean baseline.
        seed: Required for every random selector (guide §1.2).
        exclude_clean_top: Confound guard size. **Do not set to 0** except as a
            deliberate, logged demonstration of the artifact it prevents.
        mode: Projection mode; see :func:`project_out`.
        positions: Token positions to ablate at; ``None`` means all.
    """

    layers: tuple[int, ...]
    k: int
    selector: Selector = "topk"
    seed: int | None = None
    exclude_clean_top: int = 10
    mode: str = "subspace"
    positions: tuple[int, ...] | None = None

    def __post_init__(self) -> None:
        if self.selector in ("random_lens", "random_iso") and self.seed is None:
            raise ValueError(
                "random selectors require an explicit seed — an unseeded "
                "matched-random baseline is not reproducible and proposal 4.8 "
                "results computed against it are not reportable"
            )

    def key(self) -> str:
        import hashlib, json
        blob = json.dumps(asdict(self), sort_keys=True, default=str)
        return hashlib.sha256(blob.encode()).hexdigest()[:16]


@dataclass
class AblationResult:
    logits: torch.Tensor              # [n_positions, vocab] ablated next-token logits
    clean_logits: torch.Tensor        # [n_positions, vocab] unablated
    effective_rank: dict[int, torch.Tensor] = field(default_factory=dict)
    spec: AblationSpec | None = None


def prepare_lens(lens: JacobianLens, device, dtype) -> JacobianLens:
    """Move the Jacobians onto the compute device and into the model's dtype.

    Two reasons this is not optional:

    1. ``JacobianLens`` has no ``.to()``. ``transport()`` calls
       ``self.jacobians[layer].to(residual.device)`` per call, so leaving the
       lens on CPU silently copies a ``[d_model, d_model]`` matrix across the
       PCIe bus on every layer of every condition.
    2. Published lenses are stored fp16 (``save(dtype=torch.float16)``). A model
       loaded in bfloat16 will raise on ``residual @ J.T`` for mismatched
       dtypes. Neither shows up on a float32 toy model.

    Mutates and returns the lens.
    """
    lens.jacobians = {k: v.to(device=device, dtype=dtype)
                      for k, v in lens.jacobians.items()}
    return lens


@dataclass
class PromptCache:
    """Per-prompt work that every condition would otherwise repeat.

    The clean forward pass and the lens readout are identical across all
    conditions for a given prompt — only the direction *selection* differs. A
    37-condition sweep without this recomputes both 37 times.

    What is cached is deliberately small: the ranked token ids per layer, not
    the lens logits themselves. Lens logits are ``[n_positions, vocab]``, which
    at a 150k vocabulary is megabytes per layer per prompt; the ranked ids are
    ``[n_positions, k_max]``. Any ``k <= k_max`` is then a slice.

    ``k_max`` must be at least ``2 * max(k)`` in the sweep, because the
    ``next_k`` selector reads ranks ``k..2k``.
    """

    ids: torch.Tensor
    n_pos: int
    clean_logits: torch.Tensor
    excluded_ids: torch.Tensor | None          # [n_pos, n_exclude]
    ranked_ids: dict[int, torch.Tensor]        # layer -> [n_pos, k_max]
    k_max: int


@torch.no_grad()
def build_cache(
    model: Any, lens: JacobianLens, prompt: str, layers: Sequence[int],
    *, k_max: int, exclude_clean_top: int = 10, max_seq_len: int = 512,
) -> PromptCache:
    """Run the clean pass once and rank directions once, for reuse."""
    ids = model.encode(prompt, max_length=max_seq_len)
    final = model.n_layers - 1
    record_at = sorted({*layers, final})

    with ActivationRecorder(model.layers, record_at) as rec:
        model.forward(ids)
        acts = {i: rec.activations[i][0].detach() for i in record_at}
    clean_logits = model.unembed(acts[final])

    excluded = (clean_top_mask(clean_logits, exclude_clean_top)
                if exclude_clean_top > 0 else None)
    excluded_ids = (clean_logits.topk(exclude_clean_top, dim=-1).indices
                    if exclude_clean_top > 0 else None)

    ranked = {}
    for layer in layers:
        lens_logits = model.unembed(lens.transport(acts[layer], layer))
        ranked[layer] = select_by_rank(lens_logits, k_max, excluded=excluded)

    return PromptCache(ids, ids.shape[1], clean_logits, excluded_ids, ranked, k_max)


class _Ablator:
    """Forward hooks that project out precomputed per-position direction sets."""

    def __init__(
        self,
        blocks: Sequence[torch.nn.Module],
        bases: dict[int, Basis],
        position_mask: torch.Tensor | None,
        mode: str,
    ) -> None:
        self._blocks, self._bases = blocks, bases
        self._position_mask, self._mode = position_mask, mode
        self._handles: list[Any] = []

    def _hook(self, index: int):
        basis = self._bases[index]

        def fn(module, inputs, output):
            is_tuple = not torch.is_tensor(output)
            tensor = output[0] if is_tuple else output
            # tensor: [batch, seq, d_model]; harness runs batch=1.
            h = tensor[0]
            new = project_out(h, basis, mode=self._mode)
            if self._position_mask is not None:
                new = torch.where(self._position_mask[:, None], new, h)
            tensor = torch.cat([new[None], tensor[1:]], dim=0)
            return (tensor, *output[1:]) if is_tuple else tensor

        return fn

    def __enter__(self):
        try:
            for i in self._bases:
                self._handles.append(self._blocks[i].register_forward_hook(self._hook(i)))
        except Exception:
            self.__exit__()
            raise
        return self

    def __exit__(self, *exc) -> None:
        for h in self._handles:
            h.remove()
        self._handles = []


@torch.no_grad()
def run_ablation(
    model: Any,
    lens: JacobianLens,
    unembed_weight: torch.Tensor,
    prompt: str,
    spec: AblationSpec,
    *,
    max_seq_len: int = 512,
    cache: PromptCache | None = None,
) -> AblationResult:
    """Run one ablation condition end to end.

    Args:
        model: Anything satisfying ``jlens.protocol.LensModel``.
        lens: Fitted lens. ``spec.layers`` must be a subset of its source layers
            for lens-based selectors.
        unembed_weight: ``W_U``, ``[vocab, d_model]`` — usually
            ``model.lm_head.weight``. Passed explicitly because the LensModel
            protocol exposes ``unembed()`` (norm + head) but not ``W_U`` itself.
    """
    final = model.n_layers - 1

    # --- Pass 1: clean (skipped entirely when a cache is supplied) ---
    if cache is not None:
        if spec.k * (2 if spec.selector == "next_k" else 1) > cache.k_max:
            raise ValueError(
                f"cache holds k_max={cache.k_max} ranked directions but this "
                f"condition needs {spec.k * (2 if spec.selector == 'next_k' else 1)}. "
                "Rebuild the cache with a larger k_max."
            )
        ids, n_pos = cache.ids, cache.n_pos
        clean_logits, acts = cache.clean_logits, None
    else:
        ids = model.encode(prompt, max_length=max_seq_len)
        n_pos = ids.shape[1]
        with ActivationRecorder(model.layers, sorted({*spec.layers, final})) as rec:
            model.forward(ids)
            acts = {i: rec.activations[i][0].detach() for i in record_at_or(spec, final)}
        clean_logits = model.unembed(acts[final])

    if spec.selector == "none":
        return AblationResult(clean_logits, clean_logits, spec=spec)

    excluded = None
    if spec.exclude_clean_top > 0:
        excluded = (
            _mask_from_ids(cache.excluded_ids, clean_logits.shape)
            if cache is not None
            else clean_top_mask(clean_logits, spec.exclude_clean_top)
        )

    # --- Direction selection, per band layer ---
    gen = torch.Generator(device="cpu")
    if spec.seed is not None:
        gen.manual_seed(spec.seed)
    bases: dict[int, Basis] = {}
    ref = clean_logits
    for layer in spec.layers:
        h = acts[layer] if acts is not None else ref
        if spec.selector == "random_iso":
            vecs = random_isotropic(
                n_pos, spec.k, model.d_model, gen, device=h.device, dtype=h.dtype
            )
        else:
            ranked = cache.ranked_ids[layer] if cache is not None else None
            if ranked is None:
                lens_logits = model.unembed(lens.transport(h, layer))
            if spec.selector == "topk":
                tok = (ranked[:, : spec.k] if ranked is not None
                       else select_by_rank(lens_logits, spec.k, excluded=excluded))
            elif spec.selector == "next_k":
                tok = (ranked[:, spec.k : 2 * spec.k] if ranked is not None
                       else select_by_rank(lens_logits, spec.k,
                                           rank_offset=spec.k, excluded=excluded))
            elif spec.selector == "random_lens":
                tok = random_lens_tokens(
                    n_pos, spec.k, clean_logits.shape[-1], gen,
                    excluded=excluded, device=clean_logits.device,
                )
            else:
                raise ValueError(f"unknown selector {spec.selector!r}")
            vecs = lens_vectors(unembed_weight, lens.jacobians[layer].to(h.device), tok)
        bases[layer] = orthonormalise(vecs)

    pos_mask = None
    if spec.positions is not None:
        pos_mask = torch.zeros(n_pos, dtype=torch.bool, device=ids.device)
        pos_mask[list(spec.positions)] = True

    # --- Pass 2: ablated ---
    with _Ablator(model.layers, bases, pos_mask, spec.mode):
        with ActivationRecorder(model.layers, [final]) as rec2:
            model.forward(ids)
            ablated_final = rec2.activations[final][0].detach()

    return AblationResult(
        logits=model.unembed(ablated_final),
        clean_logits=clean_logits,
        effective_rank={l: b.rank for l, b in bases.items()},
        spec=spec,
    )


def greedy_match(result: AblationResult, answer_id: int, position: int = -1) -> dict:
    """Score one prompt: did the greedy next token match, clean and ablated?

    The Control A metric per DECISION_control_A §4.4 — greedy next-token
    accuracy against probe-swap.json's ``answer`` field.
    """
    return {
        "clean_correct": int(result.clean_logits[position].argmax()) == answer_id,
        "ablated_correct": int(result.logits[position].argmax()) == answer_id,
    }

## Cell 5 — Write `headroom_full.py` and `rescore.py`

In [ ]:
%%writefile headroom_full.py
"""Headroom check + diagnostic in ONE process.

Combines what were previously two runs (headroom_check.py via !python, and an
in-notebook diagnostic cell). Running both in one session loaded the model
twice and exhausted a T4. One script, one model load, three output files.

Scoring is UNCHANGED from the Qwen3.5-4B run:
  - prompt.rstrip(), answer = " " + answer.strip()   (the trailing-space fix)
  - strict := greedy continuation truncated to the answer's token length
Amendment 001's rules are NOT applied here. rescore.py stays a separate,
frozen, auditable step operating on diagnostic_rows.json.

Usage:
    python headroom_full.py --model Qwen/Qwen3-4B --data <probe-swap.json> \
        --out results/raw/headroom_qwen3-4b/ --dtype float16
"""
from __future__ import annotations
import argparse, json, subprocess
from collections import Counter, defaultdict
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

EXTRA_TOKENS = 6


def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * ((p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5) / d
    return (max(0.0, c - h), min(1.0, c + h))


@torch.no_grad()
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--dtype", default="float16")
    ap.add_argument("--limit", type=int, default=None)
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(args.model)
    model = AutoModelForCausalLM.from_pretrained(
        args.model, dtype=getattr(torch, args.dtype), device_map=device
    ).eval()

    items = json.load(open(args.data))["items"]
    if args.limit:
        items = items[: args.limit]

    rows = []
    for i, it in enumerate(items):
        prompt = it["prompt"].rstrip()
        answer = " " + it["answer"].strip()
        ans_ids = tok(answer, add_special_tokens=False).input_ids
        ids = tok(prompt, return_tensors="pt").input_ids.to(device)

        out = model.generate(
            ids, max_new_tokens=len(ans_ids) + EXTRA_TOKENS,
            do_sample=False, num_beams=1, pad_token_id=tok.eos_token_id,
        )
        gen_ids = out[0, ids.shape[1]:].tolist()
        gen_full = tok.decode(gen_ids)
        gen_trunc = tok.decode(gen_ids[: len(ans_ids)])

        a = answer.strip().lower()
        g = gen_full.strip().lower()
        r = {
            "name": it["name"], "category": it["category"], "answer": it["answer"],
            "n_answer_tokens": len(ans_ids),
            "generated": gen_trunc, "generated_full": gen_full,
            "strict": gen_trunc.strip().lower() == a,
            "exact": gen_trunc.strip().lower() == a,   # alias: same criterion
            "first_token": bool(gen_ids) and gen_ids[0] == ans_ids[0],
            "prefix": g.startswith(a),
            "window": a in g,
        }
        rows.append(r)
        print(f"[{i+1:>2}/{len(items)}] {it['name']:<26} strict={r['strict']:d} "
              f"first={r['first_token']:d}  want={it['answer']!r} got={gen_full!r}")

    n = len(rows)
    k_strict = sum(r["strict"] for r in rows)
    k_first = sum(r["first_token"] for r in rows)
    by_cat = defaultdict(list)
    for r in rows:
        by_cat[r["category"]].append(r["strict"])

    summary = {
        "model": args.model, "n": n,
        "exact": {"k": k_strict, "acc": k_strict / n, "wilson95": wilson(k_strict, n)},
        "first_token": {"k": k_first, "acc": k_first / n, "wilson95": wilson(k_first, n)},
        "prefix": {"k": sum(r["prefix"] for r in rows)},
        "window": {"k": sum(r["window"] for r in rows)},
        "answer_token_lengths": dict(Counter(r["n_answer_tokens"] for r in rows)),
        "by_category_n_ge_4": {
            c: {"n": len(v), "acc": sum(v) / len(v)}
            for c, v in sorted(by_cat.items()) if len(v) >= 4
        },
        "git_commit": subprocess.run(["git", "rev-parse", "HEAD"],
                                     capture_output=True, text=True).stdout.strip() or "UNKNOWN",
        "config": vars(args),
    }

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "rows.json").write_text(json.dumps(rows, indent=2))
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    (out_dir / "diagnostic_rows.json").write_text(json.dumps(rows, indent=2))

    lo, hi = summary["exact"]["wilson95"]
    print("\n" + "=" * 66)
    print(f"strict / exact : {k_strict}/{n} = {k_strict/n:.1%}   95% CI {lo:.1%}-{hi:.1%}")
    print(f"first_token    : {k_first}/{n} = {k_first/n:.1%}")
    print("=" * 66)
    print("Compare `strict` against the 70% threshold committed 2026-07-27.")
    print("Then run rescore.py on diagnostic_rows.json for the amended number.")
    print("\nby category (n>=4):")
    for c, v in summary["by_category_n_ge_4"].items():
        print(f"   {c:<20} {v['acc']:>6.0%}  (n={v['n']})")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile rescore.py
"""Re-score headroom generations under documented normalisation rules.

Reads the ALREADY-SAVED generations from the diagnostic run. No GPU, no model,
no regeneration — the continuations are fixed and this only changes how they are
graded, which keeps the amendment auditable.

See preregistration/amendments.md. The rules below were written AFTER seeing the
strict-miss list; that is stated in the amendment and is the reason both numbers
must be reported.
"""
from __future__ import annotations
import json, re, sys

NUM = {"zero":"0","one":"1","two":"2","three":"3","four":"4","five":"5","six":"6",
       "seven":"7","eight":"8","nine":"9","ten":"10","eleven":"11","twelve":"12",
       "thirteen":"13","fourteen":"14","fifteen":"15","sixteen":"16","seventeen":"17",
       "eighteen":"18","nineteen":"19","twenty":"20"}
ARTICLES = {"a", "an", "the"}
#: Modifiers that REVERSE or distance the meaning. R3 credits a two-word phrase
#: whose head is the answer ("honey bee" -> "bee"), but " not red" is also two
#: words with head "red" and must never be credited. Found by adversarial test,
#: not by the dataset — no negation appears in the 90 items, but ablated output
#: in Control A plausibly will.
NEGATORS = {"not", "no", "never", "non", "un", "without", "except", "besides",
            "other", "another", "different", "opposite", "unlike"}


def answer_phrase(gen: str) -> str:
    """The model's answer, cut at the first sentence end or template header."""
    s = gen.strip()
    s = re.split(r"[.\n?!;,]", s)[0]
    return s.strip().strip("'\"").lower()


def norm_number(tok: str) -> str:
    """R1 — numeral/number-word equivalence. Applies in both directions."""
    return NUM.get(tok, tok)


def strip_articles(words: list[str]) -> list[str]:
    """R2 — drop a leading determiner."""
    return words[1:] if words and words[0] in ARTICLES else words


def score(answer: str, gen: str, strict: bool) -> tuple[bool, str]:
    """Returns (credited, which_rule).

    MONOTONE BY CONSTRUCTION: a strict pass is always credited. The rules can
    only add. This is not a stylistic choice — the phrase-level rules are not a
    superset of first-token matching (the key `North` is a strict pass against
    " North America", but the phrase is "north america"), so without this an
    "amendment" would silently remove items it was never meant to touch.
    """
    a = answer.strip().lower()
    if strict:
        return True, "strict"
    phrase = answer_phrase(gen)
    if phrase == a:
        return True, "strict"

    words = strip_articles(phrase.split())
    if not words:
        return False, "-"

    # R2: article-stripped exact match
    if " ".join(words) == a:
        return True, "R2 article"

    # R1: numeral <-> word, single token only
    if len(words) == 1 and norm_number(words[0]) == norm_number(a):
        return True, "R1 numeral"

    # R3: two-word compound whose HEAD is the answer ("honey bee" -> "bee").
    # Capped at two words to keep the blast radius small; a longer phrase is a
    # different answer, not a compound form of this one.
    if len(words) == 2 and words[-1] == a and words[0] not in NEGATORS:
        return True, "R3 compound"

    return False, "-"


def main(path: str) -> None:
    rows = json.load(open(path))
    n = len(rows)
    strict = sum(r["strict"] for r in rows)
    credited, flips = 0, []
    for r in rows:
        ok, rule = score(r["answer"], r["generated_full"], r["strict"])
        credited += ok
        if ok and not r["strict"]:
            flips.append((r["name"], r["answer"], r["generated_full"], rule))
        r["amended"], r["rule"] = ok, rule

    def wilson(k, m, z=1.96):
        p = k / m; d = 1 + z**2 / m
        c = (p + z**2 / (2*m)) / d
        h = z * ((p*(1-p)/m + z**2/(4*m**2))**0.5) / d
        return max(0, c-h), min(1, c+h)

    print(f"strict  : {strict}/{n} = {strict/n:.1%}  CI {wilson(strict,n)[0]:.1%}-{wilson(strict,n)[1]:.1%}")
    print(f"amended : {credited}/{n} = {credited/n:.1%}  CI {wilson(credited,n)[0]:.1%}-{wilson(credited,n)[1]:.1%}")
    print(f"\n{len(flips)} items flipped by the rules:")
    for name, ans, gen, rule in flips:
        print(f"  [{rule:<12}] {name:<26} want={ans!r:<14} got={gen.strip()[:28]!r}")

    json.dump(rows, open("rescored_rows.json", "w"), indent=2)
    print("\nwrote rescored_rows.json")


if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else "diagnostic_rows.json")

## Cell 6 — Verify Qwen3-8B before spending anything

All four checks must pass. If any fails, stop and report back — do not run the rest.

In [ ]:
from transformers import AutoConfig
from huggingface_hub import hf_hub_download
import torch, json

HF_MODEL  = "Qwen/Qwen3-8B"
LENS_DIR  = "qwen3-8b/jlens/Salesforce-wikitext"
LENS_FILE = "Qwen3-8B_jacobian_lens.pt"
REPO      = "neuronpedia/jacobian-lens"

def nested(d, key):
    if key in d: return d[key], "top"
    for k, v in d.items():
        if isinstance(v, dict) and key in v: return v[key], k
    return None, None

cfg = AutoConfig.from_pretrained(HF_MODEL).to_dict()
arch = cfg.get("architectures")
lt, _ = nested(cfg, "layer_types")
n_layers, _ = nested(cfg, "num_hidden_layers")
d_model, _ = nested(cfg, "hidden_size")

print("architectures :", arch)
print("layer_types   :", "absent" if lt is None else dict(__import__("collections").Counter(lt)))
print("n_layers      :", n_layers)
print("d_model       :", d_model)

yml = open(hf_hub_download(REPO, filename=f"{LENS_DIR}/config.yaml")).read()
for line in yml.split("\n"):
    if any(t in line for t in ("hf_model_name", "target_layer", "prompts_fitted", "dim_batch")):
        print("   ", line.strip())

ck = torch.load(hf_hub_download(REPO, filename=f"{LENS_DIR}/{LENS_FILE}"),
                map_location="cpu", weights_only=True)
src = sorted(ck["source_layers"])
print(f"\nlens: n_prompts={ck['n_prompts']} d_model={ck['d_model']} "
      f"source_layers={src[0]}..{src[-1]}")

uniform = lt is None or len(set(lt)) == 1
is_vlm  = bool(arch) and any("ConditionalGeneration" in a or "Vision" in a for a in arch)
print("\n--- VERDICT ---")
print(f"  1. uniform layer types : {'PASS' if uniform else 'FAIL'}")
print(f"  2. not a VLM           : {'FAIL' if is_vlm else 'PASS'}")
print(f"  3. lens model matches  : {'PASS' if f'\"{HF_MODEL}\"' in yml else 'FAIL'}")
print(f"  4. d_model matches     : {'PASS' if ck['d_model']==d_model else 'FAIL'}")

## Cell 7 — Headroom smoke test, 5 prompts

Downloads the model (~16 GB, a few minutes). Check the generations look like attempted answers before running all 90.

**Edit `--dtype` to match what Cell 1 told you.**

In [ ]:
!python headroom_full.py \
    --model Qwen/Qwen3-8B \
    --data jacobian-lens/data/experiments/probe-swap.json \
    --out results/smoke_qwen3-8b/ \
    --dtype bfloat16 \
    --limit 5

## Cell 8 — Headroom, all 90

In [ ]:
!python headroom_full.py \
    --model Qwen/Qwen3-8B \
    --data jacobian-lens/data/experiments/probe-swap.json \
    --out results/raw/headroom_qwen3-8b/ \
    --dtype bfloat16

## Cell 9 — Amended score (frozen Amendment 001 rules)

In [ ]:
!python rescore.py results/raw/headroom_qwen3-8b/diagnostic_rows.json

## Cell 10 — Write the timing probe

In [ ]:
%%writefile timing_probe.py
"""Measure what Control A actually costs, before committing units to it.

Runs a handful of real ablation conditions on the real model and times them.
Everything runs in this one process, so the model is loaded once.

Nothing here is a result. No number produced by this script bears on H1, H2 or
H3 — it is a cost measurement, and the ablations it runs are thrown away.

Usage:
    python timing_probe.py --model Qwen/Qwen3-8B \
        --lens-file qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt \
        --data jacobian-lens/data/experiments/probe-swap.json
"""
from __future__ import annotations
import argparse, json, time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import jlens
from jlens.lens import JacobianLens
from ablation.harness import AblationSpec, build_cache, prepare_lens, run_ablation

LENS_REPO = "neuronpedia/jacobian-lens"


def timed(fn, *, warmup: int = 1, repeats: int = 3):
    """Median of `repeats` runs after `warmup`. Medians, not means — a single
    slow first call (allocator warmup, kernel autotuning) would otherwise
    dominate and inflate the estimate."""
    for _ in range(warmup):
        fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    ts = []
    for _ in range(repeats):
        t = time.perf_counter()
        fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        ts.append(time.perf_counter() - t)
    return sorted(ts)[len(ts) // 2]


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--lens-file", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--dtype", default="bfloat16")
    ap.add_argument("--n-band-layers", type=int, default=12,
                    help="width of the workspace band; a guess until Stage B measures it")
    ap.add_argument("--k", type=int, default=10)
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
        print("GPU:", torch.cuda.get_device_name(0))

    print("loading model ...")
    t0 = time.perf_counter()
    hf = AutoModelForCausalLM.from_pretrained(
        args.model, dtype=getattr(torch, args.dtype), device_map=device
    )
    tok = AutoTokenizer.from_pretrained(args.model)
    lm = jlens.from_hf(hf, tok)
    print(f"  model loaded in {time.perf_counter()-t0:.0f}s  "
          f"n_layers={lm.n_layers} d_model={lm.d_model}")
    if device == "cuda":
        print(f"  GPU after model: {torch.cuda.memory_allocated()/1e9:.1f} GB")

    print("loading lens ...")
    t0 = time.perf_counter()
    lens = JacobianLens.from_pretrained(LENS_REPO, filename=args.lens_file)
    lens = prepare_lens(lens, device, getattr(torch, args.dtype))
    print(f"  lens loaded in {time.perf_counter()-t0:.0f}s  "
          f"source_layers {min(lens.source_layers)}..{max(lens.source_layers)}")
    if device == "cuda":
        print(f"  GPU after lens : {torch.cuda.memory_allocated()/1e9:.1f} GB")

    wu = lm._lm_head.weight.detach()
    prompt = json.load(open(args.data))["items"][0]["prompt"].rstrip()

    # A plausible band, centred. Stage B replaces this with a measured one; the
    # timing barely depends on WHERE the band sits, only on how wide it is.
    mid = lm.n_layers // 2
    half = args.n_band_layers // 2
    band = tuple(range(max(0, mid - half), min(max(lens.source_layers) + 1, mid + half)))
    print(f"\nprobe band: layers {band[0]}..{band[-1]} ({len(band)} layers), k={args.k}")

    print("\n--- timings (median of 3) ---")
    t_cache = timed(lambda: build_cache(lm, lens, prompt, band,
                                        k_max=2 * args.k, max_seq_len=128))
    print(f"  build_cache (per prompt)     : {t_cache:7.3f} s")

    cache = build_cache(lm, lens, prompt, band, k_max=2 * args.k, max_seq_len=128)

    specs = {
        "topk (cached)":       AblationSpec(layers=band, k=args.k, selector="topk"),
        "next_k (cached)":     AblationSpec(layers=band, k=args.k, selector="next_k"),
        "random_iso (cached)": AblationSpec(layers=band, k=args.k, selector="random_iso", seed=1),
        "random_lens (cached)":AblationSpec(layers=band, k=args.k, selector="random_lens", seed=1),
    }
    times = {}
    for label, spec in specs.items():
        times[label] = timed(lambda s=spec: run_ablation(lm, lens, wu, prompt, s, cache=cache))
        print(f"  {label:<28} : {times[label]:7.3f} s")

    t_uncached = timed(lambda: run_ablation(lm, lens, wu, prompt, specs["topk (cached)"],
                                            max_seq_len=128))
    print(f"  topk (UNCACHED, for contrast): {t_uncached:7.3f} s"
          f"   -> caching saves {t_uncached / max(times['topk (cached)'], 1e-9):.1f}x")

    if device == "cuda":
        print(f"\n  peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

    # ---- extrapolation -------------------------------------------------
    mean_cond = sum(times.values()) / len(times)
    n_prompts, n_conditions = 90, 73
    total_s = n_prompts * (t_cache + n_conditions * mean_cond)
    print("\n" + "=" * 60)
    print("EXTRAPOLATION to the full Control A sweep")
    print("=" * 60)
    print(f"  grid            : {n_conditions} conditions x {n_prompts} prompts")
    print(f"  per prompt      : {t_cache:.2f}s cache + {n_conditions} x {mean_cond:.2f}s")
    print(f"  TOTAL           : {total_s/60:.0f} min  ({total_s/3600:.2f} h)")
    print()
    print("  Multiply the hours by your GPU's compute-unit rate.")
    print("  From AI guide 1.2: T4 ~1.96 units/h, A100 ~15 units/h.")
    print("  Check Colab's billing page for the L4 rate — it is not in the guide.")
    print()
    print("  If that is more than about half a month's units, cut the GRID")
    print("  (fewer random draws, fewer k values, narrower band) and record")
    print("  the cut in prereg_phase3.md BEFORE running. Not mid-sweep.")


if __name__ == "__main__":
    main()

## Cell 11 — Time Control A

Loads the model and the pre-fitted lens, then times real ablation conditions. Runs as a separate process so the headroom model is not still resident.

Nothing here is a result — the ablations are thrown away. It is a cost measurement only.

In [ ]:
!python timing_probe.py \
    --model Qwen/Qwen3-8B \
    --lens-file qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt \
    --data jacobian-lens/data/experiments/probe-swap.json \
    --dtype bfloat16 \
    --n-band-layers 12 \
    --k 10

## Cell 12 — Download everything

In [ ]:
from google.colab import files

for f in ["summary.json", "rows.json", "diagnostic_rows.json"]:
    files.download(f"results/raw/headroom_qwen3-8b/{f}")
files.download("rescored_rows.json")

---
## Report back

Paste:

1. **Cell 1** — which GPU you got
2. **Cell 6** — the four-line verdict
3. **Cell 8** — the strict/first_token summary and category breakdown
4. **Cell 9** — the amended number
5. **Cell 11** — the whole timing block, especially the extrapolation

Then: commit the JSONs, log the session, and sign the model decision.

**Still not G0.** G0 asks whether Control A passed. Control A has not run — this session only measures whether it *can* run usefully and what it would cost.